In [163]:
'''os, json — standard library
dotenv.load_dotenv — load environment variables
langchain_openai: ChatOpenAI, OpenAIEmbeddings — LLM + embeddings
langchain_pinecone.PineconeVectorStore — vector store integration
pinecone.Pinecone — Pinecone client
langchain_core.documents.Document — document schema
langchain_text_splitters.RecursiveCharacterTextSplitter — chunking text
langchain_core.prompts: ChatPromptTemplate, PromptTemplate — prompt templates
This is an OpenAI + Pinecone RAG stack built with LangChain.'''


import os
import json
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_pinecone import PineconeVectorStore
from pinecone import Pinecone
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate

In [164]:
load_dotenv()

True

In [165]:
# Load the ShopEasy knowledge base from JSON and wrap each entry in a Document.
with open("policies.json") as f:
    kb = json.load(f)

docs = [
    Document(page_content=entry["content"], metadata=entry["metadata"])
    for entry in kb
]

In [166]:
docs[2].metadata

{'department_area': 'Human Resources',
 'policy_type': 'hr_compliance',
 'employee_level': 'IC',
 'status': 'active',
 'title': 'Code of Conduct'}

In [167]:
def wrap_text(text, width=80):
    return '\n'.join([text[i:i+width] for i in range(0, len(text), width)])

for doc in docs:
    print(wrap_text(doc.page_content))
    print("-"*100)

Leave and Time-Off Policy

Annual Leave:
Full-time employees accrue 18 days of p
aid annual leave per calendar year, credited monthly at 1.5 days/month. Unused l
eave up to 5 days may be carried over to the next year; any balance beyond that 
is forfeited on December 31.

Sick Leave:
Employees receive 10 days of paid sick
 leave per year. A medical certificate is required for absences longer than 2 co
nsecutive days.

Parental Leave:
- Primary caregivers are entitled to 16 weeks o
f paid parental leave.
- Secondary caregivers are entitled to 4 weeks of paid pa
rental leave.
- Leave must be taken within 12 months of the child's birth or ado
ption.

Public Holidays:
Employees receive all locally observed public holidays 
as paid days off, as published in the regional holiday calendar each January.

R
equesting Leave:
All leave requests must be submitted through the HR portal at l
east 5 business days in advance, except for sick leave, which may be reported on
 the day of absence to the e

In [168]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,  # chunk size (characters)
    chunk_overlap=100,  # chunk overlap (characters)
    add_start_index=True,  # track index in original document
)

splits = text_splitter.split_documents(docs)

In [169]:

print(f"Documents in:  {len(docs)}")
print(f"Chunks out:    {len(splits)}")
print("-" * 100)

# Find the longest document and show how it got split (with overlap + start_index).
longest_idx = max(range(len(docs)), key=lambda i: len(docs[i].page_content))
parent = docs[longest_idx]
print(f'Longest doc: "{parent.metadata["title"]}" ({len(parent.page_content)} chars)\n')

child_chunks = [
    s for s in splits if s.metadata.get("title") == parent.metadata["title"]
]
print(f"It split into {len(child_chunks)} chunks:\n")
for i, chunk in enumerate(child_chunks):
    start = chunk.metadata["start_index"]
    print(f"[chunk {i} | start_index={start} | {len(chunk.page_content)} chars]")
    print(chunk.page_content)
    print("-" * 100)

Documents in:  51
Chunks out:    154
----------------------------------------------------------------------------------------------------
Longest doc: "IT Security and Acceptable Use Policy" (1368 chars)

It split into 4 chunks:

[chunk 0 | start_index=0 | 323 chars]
IT Security and Acceptable Use Policy

Passwords:
- Passwords must be at least 12 characters and include uppercase, lowercase, numbers, and symbols.
- Passwords must be changed every 90 days and may not be reused for the previous 5 password cycles.
- Multi-factor authentication (MFA) is mandatory for all company accounts.
----------------------------------------------------------------------------------------------------
[chunk 1 | start_index=325 | 416 chars]
Device Usage:
- Company laptops must have disk encryption and the endpoint security agent enabled at all times.
- Personal devices accessing company email or documents must be enrolled in the Mobile Device Management (MDM) system.

Software Installation:
Employees ma

In [170]:
for split in splits:
    print(wrap_text(split.page_content))
    print("-"*100)

Leave and Time-Off Policy

Annual Leave:
Full-time employees accrue 18 days of p
aid annual leave per calendar year, credited monthly at 1.5 days/month. Unused l
eave up to 5 days may be carried over to the next year; any balance beyond that 
is forfeited on December 31.

Sick Leave:
Employees receive 10 days of paid sick
 leave per year. A medical certificate is required for absences longer than 2 co
nsecutive days.
----------------------------------------------------------------------------------------------------
Parental Leave:
- Primary caregivers are entitled to 16 weeks of paid parental l
eave.
- Secondary caregivers are entitled to 4 weeks of paid parental leave.
- L
eave must be taken within 12 months of the child's birth or adoption.

Public Ho
lidays:
Employees receive all locally observed public holidays as paid days off,
 as published in the regional holiday calendar each January.
---------------------------------------------------------------------------------------------

In [171]:
#Indexing

#define the embeddings model
#ref: https://python.langchain.com/docs/integrations/text_embedding/
#text-embedding-3-small defaults to 1536 dims, but the text-embedding-3-* models
#support shortening the output via `dimensions`. We use 512 to match the index.
embeddings = OpenAIEmbeddings(model="text-embedding-3-small", dimensions=512)  # 512-dim vectors

#connect to an existing Pinecone index
#(create/manage the index in the Pinecone console; its dimension must be 512
# to match the `dimensions=512` embeddings above, with the cosine metric)
#ref: https://python.langchain.com/docs/integrations/vectorstores/pinecone/
pc = Pinecone(api_key=os.environ["PINECONE_API_KEY"])
index = pc.Index(os.environ["PINECONE_INDEX_NAME"])

#define the vector store (this notebook's data lives in the "generate_policies" namespace)
#ref: https://python.langchain.com/docs/concepts/vectorstores/
vector_store = PineconeVectorStore(
    index=index,
    embedding=embeddings,
    namespace="enterprise-company-policies",
)

In [172]:
stats = index.describe_index_stats()
existing = stats.get("namespaces", {}).get("enterprise-company-policies", {}).get("vector_count", 0)

ids = [f"{doc.metadata['title']}::{doc.metadata['start_index']}" for doc in splits]

if existing == 0:
    document_ids = vector_store.add_documents(documents=splits, ids=ids)
else:
    print(f"Index already has {existing} vectors in this namespace; skipping add.")
    document_ids = ids
document_ids

['Leave and Time-Off Policy::0',
 'Leave and Time-Off Policy::417',
 'Leave and Time-Off Policy::799',
 'Remote and Hybrid Work Policy::0',
 'Remote and Hybrid Work Policy::425',
 'Remote and Hybrid Work Policy::797',
 'Code of Conduct::0',
 'Code of Conduct::293',
 'Code of Conduct::760',
 'Code of Conduct::1133',
 'Recruitment and Hiring Policy::0',
 'Recruitment and Hiring Policy::420',
 'Recruitment and Hiring Policy::889',
 'Performance Review Policy::0',
 'Performance Review Policy::378',
 'Performance Review Policy::817',
 'Compensation and Benefits Policy::0',
 'Compensation and Benefits Policy::433',
 'Compensation and Benefits Policy::852',
 'Employee Onboarding Policy::0',
 'Employee Onboarding Policy::345',
 'Employee Onboarding Policy::688',
 'Termination and Offboarding Policy::0',
 'Termination and Offboarding Policy::416',
 'Termination and Offboarding Policy::815',
 'Anti-Harassment and Discrimination Policy::0',
 'Anti-Harassment and Discrimination Policy::293',
 'Ant

In [173]:
# Let's retrieve a record from the Pinecone index by its ID to confirm it was indexed.
# Note: Pinecone is eventually consistent, so a just-added vector may take a
# moment to become fetchable.
index.fetch(ids=[document_ids[10]], namespace="enterprise-company-policies")

FetchResponse(namespace='enterprise-company-policies', vectors={'Recruitment and Hiring Policy::0': Vector(id='Recruitment and Hiring Policy::0', values=[-0.0648803711, 0.0424499512, 0.0989379883, 0.0442199707, 0.018371582, -0.00798797607, -0.0124893188, -0.020111084, -0.0156097412, 0.0562133789, 0.0762329102, -0.099609375, 0.0257110596, -0.000164747238, 0.0347900391, -0.000912189484, -0.0445251465, -0.011390686, -0.0792236328, 0.0125656128, 0.0582580566, 0.0628662109, -0.0311279297, 0.0908813477, 0.0430297852, -0.0111618042, 0.0060005188, 0.0230255127, -0.0112609863, -0.0877685547, 0.104858398, -0.0342407227, -0.0841674805, 0.00410461426, 0.00410461426, 0.0800170898, -0.0176086426, 0.0152816772, 0.0202026367, 0.0279693604, 0.00182533264, -0.0232849121, -0.0153503418, -0.0602111816, -0.0101776123, 0.0321960449, -0.0574951172, -0.071472168, 0.0581359863, 0.0350646973, -0.0806274414, -0.0426025391, 0.0271453857, 0.00300216675, 0.0435180664, -0.00656509399, 0.0714111328, -0.0438842773, 0.

In [174]:
#configure the llm
llm = ChatOpenAI(model="gpt-4.1-mini")

#set the prompt template
template = """You are an enterprise-support assistant that helps EMPLOYEES with questions about INTERNAL COMPANY POLICIES (HR, IT, Finance, Legal, Facilities, Marketing, and Sales operations).

You do NOT have any information about external customers' orders, shipments, deliveries, tracking, charges, refunds, or returns of the company's products. If the issue is about an external customer's order, shipping, delivery, billing, or return, OR if the context below does not directly and clearly address the issue, respond with exactly:
"I don't have that information about this."

Do not guess, do not state a "likely cause", and do not try to relate unrelated policy topics to the issue.

If the context is relevant, be concise and practical: state the relevant policy and the next step the agent should take.

Context:
{context}

employee issue: {question}

Support guidance:"""

employee_rag_template = PromptTemplate.from_template(template)

In [175]:
# A sample customer support ticket (README Query 1).
user_question = "How many days are in the notice period?"

In [176]:
# Now, we'll use the vector store's similarity_search method to find the top 5 most similar chunks to the customer issue.
retrieved_docs = vector_store.similarity_search(user_question, k=5)

In [177]:
for doc in retrieved_docs:
    m = doc.metadata
    print(f"[{m['department_area']} / {m['policy_type']}] {m['title']}")
    print(doc.page_content)
    print("-" * 100)

[Human Resources / hr_onboarding] Employee Onboarding Policy
Probationary Period:
New hires are subject to a 90-day probationary period during which either party may end employment with reduced notice, as outlined in the offer letter.
----------------------------------------------------------------------------------------------------
[Human Resources / hr_leave] Leave and Time-Off Policy
Leave and Time-Off Policy

Annual Leave:
Full-time employees accrue 18 days of paid annual leave per calendar year, credited monthly at 1.5 days/month. Unused leave up to 5 days may be carried over to the next year; any balance beyond that is forfeited on December 31.

Sick Leave:
Employees receive 10 days of paid sick leave per year. A medical certificate is required for absences longer than 2 consecutive days.
----------------------------------------------------------------------------------------------------
[Human Resources / hr_offboarding] Termination and Offboarding Policy
Termination and Offboa

In [178]:
vector_store.similarity_search_with_score(user_question, k=5)

[(Document(id='Employee Onboarding Policy::688', metadata={'department_area': 'Human Resources', 'employee_level': 'IC', 'policy_type': 'hr_onboarding', 'start_index': 688.0, 'status': 'active', 'title': 'Employee Onboarding Policy'}, page_content='Probationary Period:\nNew hires are subject to a 90-day probationary period during which either party may end employment with reduced notice, as outlined in the offer letter.'),
  0.573540151),
 (Document(id='Leave and Time-Off Policy::0', metadata={'department_area': 'Human Resources', 'employee_level': 'IC', 'policy_type': 'hr_leave', 'start_index': 0.0, 'status': 'active', 'title': 'Leave and Time-Off Policy'}, page_content='Leave and Time-Off Policy\n\nAnnual Leave:\nFull-time employees accrue 18 days of paid annual leave per calendar year, credited monthly at 1.5 days/month. Unused leave up to 5 days may be carried over to the next year; any balance beyond that is forfeited on December 31.\n\nSick Leave:\nEmployees receive 10 days of pa

In [179]:
docs_content = "\n\n".join(doc.page_content for doc in retrieved_docs)
prompt = employee_rag_template.invoke({"question": user_question, "context": docs_content})
response = llm.invoke(prompt)

In [180]:
response.content

"The notice period is at least 2 weeks' written notice for regular employees and 4 weeks for management roles. During the 90-day probationary period, either party may end employment with reduced notice as outlined in the offer letter."

In [181]:
sources = [f"{doc.metadata['title']} ({doc.metadata['department_area']})" for doc in retrieved_docs]

print(f"Sources: {sources}\n\n")
print(f'Answer: {response.content}')

Sources: ['Employee Onboarding Policy (Human Resources)', 'Leave and Time-Off Policy (Human Resources)', 'Termination and Offboarding Policy (Human Resources)', 'Leave and Time-Off Policy (Human Resources)', 'Leave and Time-Off Policy (Human Resources)']


Answer: The notice period is at least 2 weeks' written notice for regular employees and 4 weeks for management roles. During the 90-day probationary period, either party may end employment with reduced notice as outlined in the offer letter.


In [182]:
retriever = vector_store.as_retriever(search_kwargs={"k": 5}, search_type='similarity')

retrieved_docs = retriever.invoke(user_question)

for doc in retrieved_docs:
    m = doc.metadata
    print(f"[{m['department_area']} / {m['policy_type']}] {m['title']}")
    print(doc.page_content)
    print("-" * 100)

[Human Resources / hr_onboarding] Employee Onboarding Policy
Probationary Period:
New hires are subject to a 90-day probationary period during which either party may end employment with reduced notice, as outlined in the offer letter.
----------------------------------------------------------------------------------------------------
[Human Resources / hr_leave] Leave and Time-Off Policy
Leave and Time-Off Policy

Annual Leave:
Full-time employees accrue 18 days of paid annual leave per calendar year, credited monthly at 1.5 days/month. Unused leave up to 5 days may be carried over to the next year; any balance beyond that is forfeited on December 31.

Sick Leave:
Employees receive 10 days of paid sick leave per year. A medical certificate is required for absences longer than 2 consecutive days.
----------------------------------------------------------------------------------------------------
[Human Resources / hr_offboarding] Termination and Offboarding Policy
Termination and Offboa

In [183]:
def generate_answer(user_question, score_threshold=0.45, debug=False):
    #retrieve the relevant docs with similarity scores
    retrieved = vector_store.similarity_search_with_score(user_question, k=5)

    if debug:
        for doc, score in retrieved:
            print(f"{score:.4f}  {doc.metadata['title']}")

    #drop chunks that aren't similar enough to be relevant
    relevant_docs = [doc for doc, score in retrieved if score >= score_threshold]

    if not relevant_docs:
        return "I don't have that information about this."

    #generate
    docs_content = "\n\n".join(doc.page_content for doc in relevant_docs)
    prompt = employee_rag_template.invoke({"question": user_question, "context": docs_content})
    response = llm.invoke(prompt)

    return response.content

generate_answer("Customer says their order never arrived even though tracking shows delivered", debug=True)

0.2389  Corporate Credit Card Policy
0.2356  Incident Response Policy
0.2334  Data Privacy Policy
0.2256  Procurement and Purchasing Policy
0.2225  IT Asset Management Policy


"I don't have that information about this."

In [184]:
generate_answer("Tell me about the company wage policy")

'The company reviews base salaries annually during the December performance review cycle. Off-cycle salary adjustments may be made for promotions, market corrections, or retention concerns with VP approval. Additionally, the company conducts an annual pay equity audit across gender and ethnicity for comparable roles and remediates any identified gaps in the next compensation cycle.'

In [185]:
generate_answer("Customer is upset about being charged but never receiving their product")

"I don't have that information about this."